In [ ]:
import pandas as pd
import glob
import re

In [ ]:
articles_path = "/Users/ajsaamirkyzy/Downloads/publications_cleaned.csv"
articles = pd.read_csv(articles_path)

# Проверим названия колонок
print("Колонки в статьях:", articles.columns)

In [ ]:
# Найти все файлы scimagojr 2001.csv ... 2025.csv
sjr_files = glob.glob("/Users/ajsaamirkyzy/Downloads/scimagojr *.csv")
print("Найдено файлов:", len(sjr_files))

sjr_list = []

# Загружаем каждый файл
for file in sjr_files:
    try:
        year = int(re.search(r'(\d{4})\.csv$', file).group(1))
        df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')
        df["Year"] = year
        sjr_list.append(df)
        print("Загружен:", year)
    except Exception as e:
        print("Ошибка в файле:", file)
        print(e)

# Объединяем все годы
sjr_all = pd.concat(sjr_list, ignore_index=True)
print("Размер объединённого SJR:", sjr_all.shape)

In [ ]:
# Очистка названий журналов
def clean_journal(name):
    if pd.isna(name):
        return name
    name = name.lower()
    name = re.sub(r'[^a-z0-9 ]', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name.strip()

articles["journal_clean"] = articles["journal"].str.strip().str.lower().apply(clean_journal)
sjr_all["Title_clean"] = sjr_all["Title"].str.strip().str.lower().apply(clean_journal)

# Убедимся, что год числовой
articles["year"] = articles["year"].astype(int)

# Merge по очищенным названиям + году
merged = articles.merge(
    sjr_all[["Title_clean", "Year", "SJR Best Quartile"]],
    left_on=["journal_clean", "year"],
    right_on=["Title_clean", "Year"],
    how="left"
)

# Проверка
print("Всего статей:", len(merged))
print("С квартилем:", merged["SJR Best Quartile"].notna().sum())
print("Без квартиля:", merged["SJR Best Quartile"].isna().sum())

# Сохраняем результат
merged.to_csv("/Users/ajsaamirkyzy/Downloads/articles_with_quartile.csv", index=False)

In [ ]:
# Фильтруем только статьи и обзоры для квартильной аналитики
journals_only = merged[merged["doc_type"].isin(["Article", "Review"])]

# Считаем количество по квартилям
quartile_counts = journals_only["SJR Best Quartile"].value_counts(dropna=False)
print(quartile_counts)

In [ ]:
# Создаём колонку для итогового квартиля
def assign_quartile(row):
    if row["doc_type"] in ["Article", "Review"]:
        return row["SJR Best Quartile"] if pd.notna(row["SJR Best Quartile"]) else "No Quartile"
    else:
        return "No Quartile"

merged["Quartile_final"] = merged.apply(assign_quartile, axis=1)

# Проверка
print("Всего статей:", len(merged))
print("Статьи с Q1–Q4:", (merged["Quartile_final"].isin(["Q1","Q2","Q3","Q4"])).sum())
print("Статьи без квартиля:", (merged["Quartile_final"]=="No Quartile").sum())

# Сохраняем итоговый CSV
merged.to_csv("/Users/ajsaamirkyzy/Downloads/articles_with_quartile_final.csv", index=False)
print("Готово! CSV сохранён с колонкой 'Quartile_final'.")

In [1]:
import pandas as pd
import glob
import re

In [2]:
articles_path = "/Users/ajsaamirkyzy/Downloads/publications_cleaned.csv"
articles = pd.read_csv(articles_path)

# Проверим названия колонок
print("Колонки в статьях:", articles.columns)


Колонки в статьях: Index(['authors', 'Author full names', 'Идентификатор автора(ов)', 'title',
       'year', 'journal', 'Том', 'Выпуск', 'Статья №', 'Страница начала',
       'Страница окончания', 'citations', 'doi', 'Ссылка', 'affiliations',
       'Авторы организаций', 'abstract', 'keywords_author', 'keywords_index',
       'funding', 'Текст о финансировании', 'sponsors', 'conference',
       'Дата конференции', 'Место проведения конференции', 'Код конференции',
       'doc_type', 'СТАДИЯ ПУБЛИКАЦИИ', 'open_access', 'Источник', 'EID',
       'area', 'open_access_bool', 'area_clean', 'n_authors', 'units',
       'kw_list', 'org_list', 'funder_list'],
      dtype='object')


In [5]:
import pandas as pd
import glob
import re

# 1️⃣ Найти все файлы scimagojr 2001.csv ... 2025.csv
sjr_files = glob.glob("/Users/ajsaamirkyzy/Downloads/scimagojr *.csv")

print("Найдено файлов:", len(sjr_files))

sjr_list = []

# 2️⃣ Загружаем каждый файл
for file in sjr_files:
    try:
        # Извлекаем год (последние 4 цифры перед .csv)
        year = int(re.search(r'(\d{4})\.csv$', file).group(1))

        df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')
        df["Year"] = year

        sjr_list.append(df)
        print("Загружен:", year)

    except Exception as e:
        print("Ошибка в файле:", file)
        print(e)

# 3️⃣ Объединяем все годы
sjr_all = pd.concat(sjr_list, ignore_index=True)

print("Размер объединённого SJR:", sjr_all.shape)

Найдено файлов: 24
Загружен: 2024
Загружен: 2018


/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/1536592804.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')
/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/1536592804.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')


Загружен: 2019
Загружен: 2009
Загружен: 2021


/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/1536592804.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')


Загружен: 2020
Загружен: 2008
Загружен: 2022
Загружен: 2023
Загружен: 2012
Загружен: 2006
Загружен: 2007
Загружен: 2013
Загружен: 2005


...

Загружен: 2011
Загружен: 2010
Загружен: 2004


/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/1536592804.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')
/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/1536592804.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')


Загружен: 2014
Загружен: 2015
Загружен: 2001


/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/1536592804.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')
/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/1536592804.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')


Загружен: 2017
Загружен: 2003
Загружен: 2002
Загружен: 2016
Размер объединённого SJR: (696583, 51)


In [ ]:

articles = pd.read_csv("/Users/ajsaamirkyzy/Downloads/publications_cleaned.csv")
# Очистка названий
articles["journal"] = articles["journal"].str.strip().str.lower()
sjr_all["Title"] = sjr_all["Title"].str.strip().str.lower()

# Убедимся, что год числовой
articles["year"] = articles["year"].astype(int)

# Merge с правильной колонкой
merged = articles.merge(
    sjr_all[["Title", "Year", "SJR Best Quartile"]],
    left_on=["journal", "year"],
    right_on=["Title", "Year"],
    how="left"
)

# Проверка
print("Всего статей:", len(merged))
print("С квартилем:", merged["SJR Best Quartile"].notna().sum())
print("Без квартиля:", merged["SJR Best Quartile"].isna().sum())

merged.to_csv(
    "/Users/ajsaamirkyzy/Downloads/articles_with_quartile.csv",
    index=False
)

Всего статей: 684
С квартилем: 348
Без квартиля: 336


print(articles.columns)

In [10]:
import re

def clean_journal(name):
    if pd.isna(name):
        return name
    name = name.lower()
    name = re.sub(r'[^a-z0-9 ]', '', name)  # оставляем только буквы и цифры
    name = re.sub(r'\s+', ' ', name)        # убираем лишние пробелы
    return name.strip()

# Создаем новые колонки
articles["journal_clean"] = articles["journal"].apply(clean_journal)
sjr_all["Title_clean"] = sjr_all["Title"].apply(clean_journal)

# Merge по очищенным названиям + году
merged_clean = articles.merge(
    sjr_all[["Title_clean", "Year", "SJR Best Quartile"]],
    left_on=["journal_clean", "year"],
    right_on=["Title_clean", "Year"],
    how="left"
)

# Проверка
print("Всего статей:", len(merged_clean))
print("С квартилем:", merged_clean["SJR Best Quartile"].notna().sum())
print("Без квартиля:", merged_clean["SJR Best Quartile"].isna().sum())


Всего статей: 684
С квартилем: 348
Без квартиля: 336


In [19]:
# Предположим, merged — это твой DataFrame с колонкой "SJR Best Quartile"
print(merged.groupby("doc_type")["SJR Best Quartile"].apply(lambda x: x.notna().sum()))

doc_type
Article             279
Book                  0
Book chapter          4
Conference paper     38
Data paper            2
Editorial             1
Erratum               2
Retracted             3
Review               19
Name: SJR Best Quartile, dtype: int64


In [20]:
# Фильтруем только статьи и обзоры для квартильной аналитики
journals_only = merged[merged["doc_type"].isin(["Article", "Review"])]

# Считаем количество по квартилям
quartile_counts = journals_only["SJR Best Quartile"].value_counts(dropna=False)
print(quartile_counts)

SJR Best Quartile
NaN    180
Q2     113
Q1      84
Q3      66
Q4      27
-        8
Name: count, dtype: int64


In [ ]:
import pandas as pd
import glob
import re


sjr_files = glob.glob("/Users/ajsaamirkyzy/Downloads/scimagojr *.csv")

print("Найдено файлов:", len(sjr_files))

sjr_list = []


for file in sjr_files:
    try:
        # Извлекаем год (последние 4 цифры перед .csv)
        year = int(re.search(r'(\d{4})\.csv$', file).group(1))

        df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')
        df["Year"] = year

        sjr_list.append(df)
        print("Загружен:", year)

    except Exception as e:
        print("Ошибка в файле:", file)
        print(e)

# 3️⃣ Объединяем все годы
sjr_all = pd.concat(sjr_list, ignore_index=True)

print("Размер объединённого SJR:", sjr_all.shape)
# 1️⃣ Загружаем статьи
articles = pd.read_csv("/Users/ajsaamirkyzy/Downloads/publications_cleaned.csv")

# 2️⃣ Загружаем объединённые данные SJR
 # твой объединённый SJR

# 3️⃣ Очистка названий журналов для merge
import re
def clean_journal(name):
    if pd.isna(name):
        return name
    name = name.lower()
    name = re.sub(r'[^a-z0-9 ]', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name.strip()

articles["journal_clean"] = articles["journal"].apply(clean_journal)


# 4️⃣ Убедимся, что год числовой
articles["year"] = articles["year"].astype(int)



# 6️⃣ Создаём колонку для итогового квартиля
# Для Article и Review используем SJR Best Quartile
# Для остальных типов — ставим 'No Quartile'
def assign_quartile(row):
    if row["doc_type"] in ["Article", "Review"]:
        return row["SJR Best Quartile"] if pd.notna(row["SJR Best Quartile"]) else "No Quartile"
    else:
        return "No Quartile"

merged["Quartile_final"] = merged.apply(assign_quartile, axis=1)

# 7️⃣ Проверка
print("Всего статей:", len(merged))
print("Статьи с Q1–Q4:", (merged["Quartile_final"].isin(["Q1","Q2","Q3","Q4"])).sum())
print("Статьи без квартиля:", (merged["Quartile_final"]=="No Quartile").sum())

# 8️⃣ Сохраняем итоговый CSV
merged.to_csv("/Users/ajsaamirkyzy/Downloads/articles_with_quartile_final.csv", index=False)

print("Готово! CSV сохранён с колонкой 'Quartile_final'.")

Найдено файлов: 24
Загружен: 2024


/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/2268965672.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')
/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/2268965672.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')


Загружен: 2018
Загружен: 2019
Загружен: 2009
Загружен: 2021
Загружен: 2020
Загружен: 2008


/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/2268965672.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')


Загружен: 2022
Загружен: 2023
Загружен: 2012
Загружен: 2006
Загружен: 2007
Загружен: 2013
Загружен: 2005


...

/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/2268965672.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')
/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/2268965672.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')


Загружен: 2014
Загружен: 2015
Загружен: 2001


/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/2268965672.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')


Загружен: 2017
Загружен: 2003
Загружен: 2002
Загружен: 2016
Размер объединённого SJR: (696583, 51)
Всего статей: 684
Статьи с Q1–Q4: 290
Статьи без квартиля: 386
Готово! CSV сохранён с колонкой 'Quartile_final'.


/var/folders/xp/lktrqh_x74x_fyy96m3qcclh0000gn/T/ipykernel_1107/2268965672.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', quotechar='"', encoding='utf-8')


In [ ]:
# Фильтруем только статьи и обзоры для квартильной аналитики
journals_only = merged[merged["doc_type"].isin(["Article", "Review"])]

# Считаем количество по квартилям
quartile_counts = journals_only["SJR Best Quartile"].value_counts(dropna=False)
print(quartile_counts)

SJR Best Quartile
NaN    180
Q2     113
Q1      84
Q3      66
Q4      27
-        8
Name: count, dtype: int64


In [28]:
# Оставляем только нужные колонки
columns_to_keep = [
    "authors", "Author full names", "Идентификатор автора(ов)", "title", "year", 
    "journal", "Том", "Выпуск", "Статья №", "Страница начала", "Страница окончания",
    "citations", "doi", "Ссылка", "affiliations", "Авторы организаций", "abstract",
    "keywords_author", "keywords_index", "funding", "Текст о финансировании", "sponsors",
    "conference", "Дата конференции", "Место проведения конференции", "Код конференции",
    "doc_type", "СТАДИЯ ПУБЛИКАЦИИ", "open_access", "Источник", "EID", "area",
    "open_access_bool", "area_clean", "n_authors", "units", "kw_list", "org_list",
    "funder_list", "Quartile_final"
]

final_df = merged[columns_to_keep]

# Сохраняем чистый CSV
final_df.to_csv("/Users/ajsaamirkyzy/Downloads/articles_with_quartile_final_clean.csv", index=False)

print("Готово! Дубли колонок удалены, сохранён чистый CSV.")

Готово! Дубли колонок удалены, сохранён чистый CSV.
